In [ ]:
# Stage 0' - SynthPose config: single source of truth for the 52-keypoint feature space.
from dataclasses import dataclass

# ==================== Paths ====================
YOGNET_ROOT      = "path/to/yognet7poses"
YOGNET_SP_OUT    = "path/to/working/extracted_yognet_synthpose"
YOGNET_SP_CLIPS  = f"{YOGNET_SP_OUT}/clips"
YOGNET_SP_META   = f"{YOGNET_SP_OUT}/metadata.csv"

# ==================== Extraction hyperparams ====================
FRAME_STRIDE     = 2     # matches BlazePose Stage 4 for fair comparison
MIN_FRAMES       = 33    # matches BlazePose Stage 4
DET_THRESHOLD    = 0.3   # RTDetr person-box confidence
VIS_THRESH       = 0.3   # SynthPose score gate (applied at Stage 2′ feature eng, NOT here)

# ==================== Clip schema ====================
N_KEYPOINTS      = 52
N_CHANNELS       = 3     # (x, y, score)  — NOT 4. No faked z-channel.

# ==================== Pose vocabulary (unchanged from existing pipeline) ====================
POSE_NAMES = ["cobra", "corpse", "downward_facing_dog",
              "mountain", "tree", "triangle", "warrior"]
POSE_TO_ID = {name: i for i, name in enumerate(POSE_NAMES)}
ID_TO_POSE = {i: name for i, name in enumerate(POSE_NAMES)}

# YogNet folder name → canonical pose name (matches existing Stage 4 mapping)
YOGNET_FOLDER_MAP = {
    "Bhujangasana":    "cobra",
    "Shavasana":       "corpse",
    "Adhomukhasvanasana": "downward_facing_dog",
    "Tadasana":        "mountain",
    "Vrikshasana":     "tree",
    "Trikonasana":     "triangle",
    "Virbhadrasana1":  "warrior",
}

# ==================== 52-keypoint topology ====================
# Order matches HF model card: 17 COCO + 35 anatomical markers.
# https://huggingface.co/stanfordmimi/synthpose-vitpose-base-hf

SYNTHPOSE_NAMES = [
    # COCO-17 (indices 0..16)
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle",
    # Anatomical markers (indices 17..51)
    "sternum",
    "rshoulder", "lshoulder",
    "r_lelbow", "l_lelbow", "r_melbow", "l_melbow",
    "r_lwrist", "l_lwrist", "r_mwrist", "l_mwrist",
    "r_ASIS", "l_ASIS", "r_PSIS", "l_PSIS",
    "r_knee", "l_knee", "r_mknee", "l_mknee",
    "r_ankle", "l_ankle", "r_mankle", "l_mankle",
    "r_5meta", "l_5meta",
    "r_toe", "l_toe", "r_big_toe", "l_big_toe",
    "l_calc", "r_calc",
    "C7", "L2", "T11", "T6",
]
assert len(SYNTHPOSE_NAMES) == N_KEYPOINTS, \
    f"Expected {N_KEYPOINTS} names, got {len(SYNTHPOSE_NAMES)}"

# Named-access namespace. Import `KP` everywhere — no more `LS = 11` magic numbers.
@dataclass(frozen=True)
class _KP:
    # COCO-17
    nose: int            = 0
    left_eye: int        = 1
    right_eye: int       = 2
    left_ear: int        = 3
    right_ear: int       = 4
    left_shoulder: int   = 5
    right_shoulder: int  = 6
    left_elbow: int      = 7
    right_elbow: int     = 8
    left_wrist: int      = 9
    right_wrist: int     = 10
    left_hip: int        = 11
    right_hip: int       = 12
    left_knee_coco: int  = 13    # renamed to avoid collision with anatomical l_knee
    right_knee_coco: int = 14
    left_ankle_coco: int = 15
    right_ankle_coco: int= 16
    # Anatomical
    sternum: int         = 17
    rshoulder: int       = 18
    lshoulder: int       = 19
    r_lelbow: int        = 20    # lateral epicondyle right
    l_lelbow: int        = 21
    r_melbow: int        = 22    # medial epicondyle right
    l_melbow: int        = 23
    r_lwrist: int        = 24    # lateral (ulnar styloid) wrist
    l_lwrist: int        = 25
    r_mwrist: int        = 26    # medial (radial styloid) wrist
    l_mwrist: int        = 27
    r_ASIS: int          = 28
    l_ASIS: int          = 29
    r_PSIS: int          = 30
    l_PSIS: int          = 31
    r_knee: int          = 32    # lateral epicondyle right knee
    l_knee: int          = 33
    r_mknee: int         = 34    # medial epicondyle right knee
    l_mknee: int         = 35
    r_ankle: int         = 36    # lateral malleolus right
    l_ankle: int         = 37
    r_mankle: int        = 38    # medial malleolus right
    l_mankle: int        = 39
    r_5meta: int         = 40
    l_5meta: int         = 41
    r_toe: int           = 42
    l_toe: int           = 43
    r_big_toe: int       = 44
    l_big_toe: int       = 45
    l_calc: int          = 46    # calcaneus (heel)
    r_calc: int          = 47
    C7: int              = 48
    L2: int              = 49
    T11: int             = 50
    T6: int              = 51

KP = _KP()

# Sanity check: named indices agree with SYNTHPOSE_NAMES order
# (left_knee_coco and right_knee_coco correspond to COCO names 'left_knee'/'right_knee')
_name_check = {
    "nose": KP.nose, "left_shoulder": KP.left_shoulder, "right_shoulder": KP.right_shoulder,
    "sternum": KP.sternum, "r_ASIS": KP.r_ASIS, "l_ASIS": KP.l_ASIS,
    "r_knee": KP.r_knee, "l_knee": KP.l_knee, "C7": KP.C7, "T6": KP.T6,
}
for nm, idx in _name_check.items():
    assert SYNTHPOSE_NAMES[idx] == nm, f"Topology mismatch: {nm} != {SYNTHPOSE_NAMES[idx]}"

# ==================== Landmark groups for Stage 2′ feature engineering ====================
# Pairs and chains you'll use for joint-angle features, normalization, and chain-rotation
# augmentation. Defined here so Stage 2′ doesn't reinvent them.

# Preferred anatomical markers (bony) — use these over COCO duplicates where both exist.
SHOULDERS = (KP.lshoulder, KP.rshoulder)        # anatomical > COCO
HIPS_ASIS = (KP.l_ASIS, KP.r_ASIS)              # front pelvis
HIPS_PSIS = (KP.l_PSIS, KP.r_PSIS)              # back pelvis
ANKLES    = (KP.l_ankle, KP.r_ankle)            # lateral malleolus
KNEES     = (KP.l_knee, KP.r_knee)              # lateral epicondyle

# Joint-center midpoints (computed in Stage 2′ from pairs)
# knee_center_L = midpoint(l_knee, l_mknee), etc.
KNEE_PAIRS  = {"L": (KP.l_knee, KP.l_mknee), "R": (KP.r_knee, KP.r_mknee)}
ANKLE_PAIRS = {"L": (KP.l_ankle, KP.l_mankle), "R": (KP.r_ankle, KP.r_mankle)}
ELBOW_PAIRS = {"L": (KP.l_lelbow, KP.l_melbow), "R": (KP.r_lelbow, KP.r_melbow)}
WRIST_PAIRS = {"L": (KP.l_lwrist, KP.l_mwrist), "R": (KP.r_lwrist, KP.r_mwrist)}

# Spine chain (useful for lumbar/thoracic angles in Stage 2′ features
# and for any Proposal-3-style follow-up work)
SPINE_CHAIN = [KP.C7, KP.T6, KP.T11, KP.L2]   # top-to-bottom

# ==================== Chain-downstream map for synthetic-incorrect augmentation ====================
# Stage 2′ rewrites the existing chain-rotation logic against the new topology.
# For each vertex, which keypoints rotate with it when we apply a chain perturbation?
# (Left arm chain: rotating the shoulder moves elbow + wrist downstream.)
# This is defined here as a constant so Stage 2′ can import it directly.

CHAIN_DOWNSTREAM = {
    # Left arm: shoulder → elbow + wrist cluster + fingers (no fingers in SynthPose, stop at wrist)
    KP.lshoulder:  [KP.left_elbow, KP.l_lelbow, KP.l_melbow,
                    KP.left_wrist, KP.l_lwrist, KP.l_mwrist],
    # Right arm
    KP.rshoulder:  [KP.right_elbow, KP.r_lelbow, KP.r_melbow,
                    KP.right_wrist, KP.r_lwrist, KP.r_mwrist],
    # Left leg: hip (via l_ASIS) → knee cluster + ankle cluster + foot
    KP.l_ASIS:     [KP.left_knee_coco, KP.l_knee, KP.l_mknee,
                    KP.left_ankle_coco, KP.l_ankle, KP.l_mankle,
                    KP.l_5meta, KP.l_toe, KP.l_big_toe, KP.l_calc],
    # Right leg
    KP.r_ASIS:     [KP.right_knee_coco, KP.r_knee, KP.r_mknee,
                    KP.right_ankle_coco, KP.r_ankle, KP.r_mankle,
                    KP.r_5meta, KP.r_toe, KP.r_big_toe, KP.r_calc],
    # Left knee (below): just the ankle + foot cluster
    KP.l_knee:     [KP.left_ankle_coco, KP.l_ankle, KP.l_mankle,
                    KP.l_5meta, KP.l_toe, KP.l_big_toe, KP.l_calc],
    KP.r_knee:     [KP.right_ankle_coco, KP.r_ankle, KP.r_mankle,
                    KP.r_5meta, KP.r_toe, KP.r_big_toe, KP.r_calc],
    # Left elbow (below): just wrist cluster
    KP.left_elbow: [KP.left_wrist, KP.l_lwrist, KP.l_mwrist],
    KP.right_elbow:[KP.right_wrist, KP.r_lwrist, KP.r_mwrist],
}

print(f"Stage 0′ loaded — SynthPose topology, {N_KEYPOINTS} keypoints, schema (T, {N_KEYPOINTS}, {N_CHANNELS})")
print(f"  Output root: {YOGNET_SP_OUT}")
print(f"  Chain-rotation vertices defined: {len(CHAIN_DOWNSTREAM)}")


In [ ]:
# Stage 1' - YogNet SynthPose extraction (RTDetr + SynthPose-ViTPose-base) -> (T, 52, 3) clips.
import os, cv2, csv, time, torch
import numpy as np
import pandas as pd
from PIL import Image
from transformers import (
    AutoProcessor,
    RTDetrForObjectDetection,
    VitPoseForPoseEstimation,
)

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}

os.makedirs(YOGNET_SP_CLIPS, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cpu":
    print("WARNING: running on CPU")

# ================= Load models once =================
print("Loading RTDetr person detector...")
det_proc  = AutoProcessor.from_pretrained("PekingU/rtdetr_r50vd_coco_o365")
det_model = RTDetrForObjectDetection.from_pretrained(
    "PekingU/rtdetr_r50vd_coco_o365"
).to(device).eval()

print("Loading SynthPose ViTPose-base...")
pose_proc  = AutoProcessor.from_pretrained("yonigozlan/synthpose-vitpose-base-hf")
pose_model = VitPoseForPoseEstimation.from_pretrained(
    "yonigozlan/synthpose-vitpose-base-hf"
).to(device).eval()

# ================= Helpers =================
def detect_largest_person(pil_img):
    """Return (x, y, w, h) of largest person box, or None."""
    inputs = det_proc(images=pil_img, return_tensors="pt").to(device)
    with torch.no_grad():
        out = det_model(**inputs)
    results = det_proc.post_process_object_detection(
        out,
        target_sizes=torch.tensor([(pil_img.height, pil_img.width)]),
        threshold=DET_THRESHOLD,
    )[0]
    # COCO class 0 = person
    person_mask  = results["labels"] == 0
    person_boxes = results["boxes"][person_mask].cpu().numpy()
    if len(person_boxes) == 0:
        return None
    # VOC (x1,y1,x2,y2) → COCO (x,y,w,h), pick largest area
    x1, y1, x2, y2 = person_boxes.T
    areas = (x2 - x1) * (y2 - y1)
    i = int(np.argmax(areas))
    return np.array([x1[i], y1[i], x2[i] - x1[i], y2[i] - y1[i]])

def extract_keypoints(pil_img, box_xywh):
    """Run SynthPose on a single cropped person, return (52, 3) array."""
    inputs = pose_proc(pil_img, boxes=[[box_xywh]], return_tensors="pt").to(device)
    with torch.no_grad():
        out = pose_model(**inputs)
    pose_results = pose_proc.post_process_pose_estimation(out, boxes=[[box_xywh]])
    # pose_results[0] = list of results for image 0; [0] = first (only) person
    person = pose_results[0][0]
    kpts   = np.asarray(person["keypoints"], dtype=np.float32)   # (52, 2)
    scores = np.asarray(person["scores"],    dtype=np.float32)   # (52,)
    return np.concatenate([kpts, scores[:, None]], axis=1)       # (52, 3)

def process_video(video_path, stride=FRAME_STRIDE):
    """Extract (T, 52, 3) clip from a video."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None, None

    kept_frames  = []
    detected_cnt = 0
    frame_idx    = 0

    while True:
        ok, bgr = cap.read()
        if not ok:
            break
        if frame_idx % stride != 0:
            frame_idx += 1
            continue

        pil = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        box = detect_largest_person(pil)
        if box is None:
            # undetected frame → zeros (matches Stage 4 convention)
            kept_frames.append(np.zeros((N_KEYPOINTS, N_CHANNELS), dtype=np.float32))
        else:
            kp = extract_keypoints(pil, box)
            kept_frames.append(kp)
            detected_cnt += 1
        frame_idx += 1

    cap.release()
    if not kept_frames:
        return None, None

    clip = np.stack(kept_frames, axis=0).astype(np.float32)     # (T, 52, 3)
    T    = clip.shape[0]
    return clip, {
        "frames_kept":     T,
        "frames_detected": detected_cnt,
        "detection_rate":  round(detected_cnt / max(T, 1), 4),
    }

# ================= Main loop =================
if not os.path.isdir(YOGNET_ROOT):
    raise FileNotFoundError(f"YOGNET_ROOT does not exist: {YOGNET_ROOT}")

found_folders = sorted(
    d for d in os.listdir(YOGNET_ROOT)
    if os.path.isdir(os.path.join(YOGNET_ROOT, d))
)
print(f"YogNet folders found: {found_folders}")

metadata_rows  = []
clip_counter   = 0
skipped_unread = skipped_short = skipped_unmap = 0
t_start = time.time()

for folder in found_folders:
    pose_canonical = YOGNET_FOLDER_MAP.get(folder)
    if pose_canonical is None:
        skipped_unmap += 1
        print(f"  [skip] no canonical mapping for folder '{folder}'")
        continue

    folder_path = os.path.join(YOGNET_ROOT, folder)
    video_files = sorted(
        f for f in os.listdir(folder_path)
        if os.path.splitext(f)[1].lower() in VIDEO_EXTS
    )
    print(f"\n[{pose_canonical}] {folder}: {len(video_files)} videos")

    for vf in video_files:
        vpath = os.path.join(folder_path, vf)
        t_clip = time.time()
        clip, stats = process_video(vpath)

        if clip is None:
            skipped_unread += 1
            continue
        if stats["frames_kept"] < MIN_FRAMES:
            skipped_short += 1
            continue

        out_name = f"{pose_canonical}__correct__{clip_counter:04d}.npy"
        out_path = os.path.join(YOGNET_SP_CLIPS, out_name)
        np.save(out_path, clip)

        metadata_rows.append({
            "clip_index":      clip_counter,
            "pose":            pose_canonical,
            "original_label":  folder,
            "quality_label":   "correct",
            "frames_kept":     stats["frames_kept"],
            "frames_detected": stats["frames_detected"],
            "detection_rate":  stats["detection_rate"],
            "clip_path":       out_path,
            "source_file":     vpath,
            "split":           "",
            "dataset":         "yognet_synthpose",
        })
        clip_counter += 1
        if clip_counter % 10 == 0:
            elapsed = time.time() - t_start
            rate    = clip_counter / elapsed
            print(f"  processed {clip_counter} clips  "
                  f"({rate:.2f} clips/s, last clip {time.time()-t_clip:.1f}s)")

# ================= Write metadata =================
fieldnames = ["clip_index", "pose", "original_label", "quality_label",
              "frames_kept", "frames_detected", "detection_rate",
              "clip_path", "source_file", "split", "dataset"]
with open(YOGNET_SP_META, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(metadata_rows)

# ================= Summary =================
print("\n" + "=" * 60)
print(f"Stage 1′ complete — YogNet SynthPose extraction")
print(f"  clips saved       : {clip_counter}")
print(f"  skipped (unread)  : {skipped_unread}")
print(f"  skipped (short)   : {skipped_short}  < {MIN_FRAMES} frames")
print(f"  skipped (unmap)   : {skipped_unmap}  folder→pose mapping")
print(f"  metadata          : {YOGNET_SP_META}")
print(f"  total wall time   : {(time.time() - t_start)/60:.1f} min")

if metadata_rows:
    meta = pd.DataFrame(metadata_rows)
    print("\nClips per target pose:")
    print(meta.groupby("pose")["clip_index"].count().to_string())
    print("\nMean detection rate by pose:")
    print(meta.groupby("pose")["detection_rate"].mean().map("{:.2%}".format).to_string())
    print("\nFrames per clip:")
    print(meta["frames_kept"].describe().round(1).to_string())


In [ ]:
# Stage 1′ pickup — process folders missed by the first run due to name mismatches

MISSED = {
    "Adhomukhasvanasana": "downward_facing_dog",
    "Virbhadrasana1":     "warrior",
}

# Continue clip numbering from where the main run ended
existing = pd.read_csv(YOGNET_SP_META)
clip_counter = int(existing["clip_index"].max()) + 1 if len(existing) else 0
print(f"Existing clips on disk: {len(existing)} | resuming clip_index at {clip_counter}\n")

new_rows = []
skipped_unread = skipped_short = 0

for folder, pose_canonical in MISSED.items():
    folder_path = os.path.join(YOGNET_ROOT, folder)
    if not os.path.isdir(folder_path):
        print(f"  [warn] folder not found, skipping: {folder_path}")
        continue

    video_files = sorted(
        f for f in os.listdir(folder_path)
        if os.path.splitext(f)[1].lower() in VIDEO_EXTS
    )
    print(f"[{pose_canonical}] {folder}: {len(video_files)} videos")

    for vf in video_files:
        vpath = os.path.join(folder_path, vf)
        clip, stats = process_video(vpath)

        if clip is None:
            skipped_unread += 1
            continue
        if stats["frames_kept"] < MIN_FRAMES:
            skipped_short += 1
            continue

        out_name = f"{pose_canonical}__correct__{clip_counter:04d}.npy"
        out_path = os.path.join(YOGNET_SP_CLIPS, out_name)
        np.save(out_path, clip)

        new_rows.append({
            "clip_index":      clip_counter,
            "pose":            pose_canonical,
            "original_label":  folder,
            "quality_label":   "correct",
            "frames_kept":     stats["frames_kept"],
            "frames_detected": stats["frames_detected"],
            "detection_rate":  stats["detection_rate"],
            "clip_path":       out_path,
            "source_file":     vpath,
            "split":           "",
            "dataset":         "yognet_synthpose",
        })
        clip_counter += 1
        if len(new_rows) % 10 == 0:
            print(f"  +{len(new_rows)} clips so far")

# Append to metadata.csv
updated = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
updated.to_csv(YOGNET_SP_META, index=False)

# Also patch the in-memory dict so future cells don't re-stumble
YOGNET_FOLDER_MAP["Adhomukhasvanasana"] = "downward_facing_dog"
YOGNET_FOLDER_MAP["Virbhadrasana1"]     = "warrior"

# ==================== Summary ====================
print("\n" + "=" * 60)
print(f"Pickup complete")
print(f"  added clips      : {len(new_rows)}")
print(f"  skipped (unread) : {skipped_unread}")
print(f"  skipped (short)  : {skipped_short}")
print(f"  total on disk    : {len(updated)}")

print("\nClips per target pose (full YogNet SynthPose set):")
print(updated.groupby("pose")["clip_index"].count().to_string())
print("\nMean detection rate by pose:")
print(updated.groupby("pose")["detection_rate"].mean().map("{:.2%}".format).to_string())
print("\nFrames per clip:")
print(updated["frames_kept"].describe().round(1).to_string())


In [ ]:
# Stage 2' - feature engineering + synthetic-incorrect augmentation + windowing (52-kp).
import os, numpy as np, pandas as pd

OUT = "path/to/working/windows_synthpose"
os.makedirs(OUT, exist_ok=True)

# Windowing — identical to BlazePose Stage 2 for fair comparison
WINDOW_SIZE    = 32
STRIDE         = 32
MIN_VIS_FRAC   = 0.6

# Augmentation — identical ranges to BlazePose Stage 2
N_CORRUPT_JOINTS_MIN = 1
N_CORRUPT_JOINTS_MAX = 2
CORRUPT_ANGLE_MIN_DEG = 0.0
CORRUPT_ANGLE_MAX_DEG = 45.0
JITTER_SIGMA = 0.01

N_POSES = len(POSE_NAMES)
LABEL_CORRECT, LABEL_INCORRECT = 0, 1

# Six feedback joints — same names/order as BlazePose Stage 2 so checkpoints align
JOINT_NAMES = ["left_elbow", "right_elbow", "left_knee",
               "right_knee", "left_hip",    "right_hip"]
N_JOINTS    = 6

RNG = np.random.default_rng(42)

# Load the YogNet SynthPose metadata produced by Stage 1′
meta = pd.read_csv(YOGNET_SP_META)
print(f"Loaded {len(meta)} SynthPose clips from {YOGNET_SP_META}")
print(f"Window {WINDOW_SIZE} / stride {STRIDE} / min-vis-frac {MIN_VIS_FRAC}")
print(f"Poses ({N_POSES}): {POSE_NAMES}")
print(f"Feedback joints ({N_JOINTS}): {JOINT_NAMES}")


In [ ]:
# Normalization references: hip center = midpoint(l_ASIS, r_ASIS); scale = shoulder width.
HIP_CENTER_IDXS   = (KP.l_ASIS, KP.r_ASIS)
SHOULDER_AB_IDXS  = (KP.lshoulder, KP.rshoulder)

# ---- Feedback-joint definitions ----
# For each of the 6 feedback joints we need:
#   VERTEX      : pivot landmark for chain rotation (single index)
#   DOWNSTREAM  : all landmarks that rotate rigidly with the distal segment
#   ANGLE_TRIPLE: (a, b, c) for the 3-point angle feature (b is vertex)
#
# Pivot/angle-vertex choice: anatomical where available, COCO otherwise.
# Downstream clusters include BOTH the COCO twin and all anatomical
# (medial/lateral) satellites of the distal joint(s) so they move rigidly
# together as a chain — otherwise the corruption leaves the cluster dislocated.

JOINT_VERTEX = {
    "left_elbow":  KP.left_elbow,   # COCO idx 7
    "right_elbow": KP.right_elbow,  # COCO idx 8
    "left_knee":   KP.l_knee,       # anatomical lateral epicondyle (33)
    "right_knee":  KP.r_knee,       # anatomical (32)
    "left_hip":    KP.l_ASIS,       # anatomical (29)
    "right_hip":   KP.r_ASIS,       # anatomical (28)
}

CHAIN_DOWNSTREAM = {
    # Left arm: elbow → entire wrist cluster (COCO + anatomical medial/lateral)
    "left_elbow":  [KP.left_wrist, KP.l_lwrist, KP.l_mwrist],
    "right_elbow": [KP.right_wrist, KP.r_lwrist, KP.r_mwrist],
    # Left leg at knee: ankle cluster + foot
    "left_knee":   [KP.left_ankle_coco, KP.l_ankle, KP.l_mankle,
                    KP.l_5meta, KP.l_toe, KP.l_big_toe, KP.l_calc],
    "right_knee":  [KP.right_ankle_coco, KP.r_ankle, KP.r_mankle,
                    KP.r_5meta, KP.r_toe, KP.r_big_toe, KP.r_calc],
    # Left hip: entire leg — knee cluster + ankle cluster + foot
    "left_hip":    [KP.left_knee_coco, KP.l_knee, KP.l_mknee,
                    KP.left_ankle_coco, KP.l_ankle, KP.l_mankle,
                    KP.l_5meta, KP.l_toe, KP.l_big_toe, KP.l_calc],
    "right_hip":   [KP.right_knee_coco, KP.r_knee, KP.r_mknee,
                    KP.right_ankle_coco, KP.r_ankle, KP.r_mankle,
                    KP.r_5meta, KP.r_toe, KP.r_big_toe, KP.r_calc],
}

# 3-point angle triples (a, b=vertex, c). Anatomical wherever possible.
ANGLE_TRIPLES = {
    "left_elbow":  (KP.lshoulder,   KP.left_elbow,  KP.left_wrist),
    "right_elbow": (KP.rshoulder,   KP.right_elbow, KP.right_wrist),
    "left_knee":   (KP.l_ASIS,      KP.l_knee,      KP.l_ankle),
    "right_knee":  (KP.r_ASIS,      KP.r_knee,      KP.r_ankle),
    "left_hip":    (KP.lshoulder,   KP.l_ASIS,      KP.l_knee),
    "right_hip":   (KP.rshoulder,   KP.r_ASIS,      KP.r_knee),
}

# Feature-vector layout for SynthPose (T, 52, 3):
#   xy      :  52 * 2 = 104
#   vxy     :  52 * 2 = 104   (first-difference of xy across time)
#   score   :  52
#   angles  :  12
#   ----------------
#   TOTAL   : 272
FEAT_DIM       = 272
SCORE_SLICE    = slice(208, 260)   # score block in feature vector
VIS_THRESH     = 0.3

def normalize_clip(clip):
    """Hip-center (ASIS midpoint) + shoulder-scale per frame. clip: (T, 52, 3).
    Modifies xy only; score channel untouched."""
    out = clip.copy()
    lhip, rhip = HIP_CENTER_IDXS
    lsho, rsho = SHOULDER_AB_IDXS
    for t in range(len(out)):
        hip   = (out[t, lhip, :2] + out[t, rhip, :2]) / 2.0
        out[t, :, :2] -= hip
        scale = np.linalg.norm(out[t, lsho, :2] - out[t, rsho, :2]) + 1e-6
        out[t, :, :2] /= scale
    return out

def _angle_3pts_2d(a, b, c):
    ba = a - b; bc = c - b
    denom = (np.linalg.norm(ba) * np.linalg.norm(bc)) + 1e-8
    return float(np.degrees(np.arccos(np.clip(np.dot(ba, bc) / denom, -1.0, 1.0))))

def compute_angles(kp):
    """12-dim angle feature vector (6 joint angles + 3 L-R diffs + 3 L+R means).
    kp: (52, 3) — xy + score."""
    xy, score = kp[:, :2], kp[:, 2]
    def safe(name):
        a, b, c = ANGLE_TRIPLES[name]
        if min(score[a], score[b], score[c]) >= VIS_THRESH:
            return _angle_3pts_2d(xy[a], xy[b], xy[c])
        return 0.0
    l_el = safe("left_elbow");  r_el = safe("right_elbow")
    l_kn = safe("left_knee");   r_kn = safe("right_knee")
    l_hp = safe("left_hip");    r_hp = safe("right_hip")
    return np.array([l_el, r_el, l_kn, r_kn, l_hp, r_hp,
                     l_el - r_el, l_kn - r_kn, l_hp - r_hp,
                     (l_el + r_el)/2, (l_kn + r_kn)/2, (l_hp + r_hp)/2],
                    dtype=np.float32)

def clip_to_features(clip_norm):
    """Normalized (T, 52, 3) clip → (T, 272) feature matrix."""
    T     = len(clip_norm)
    xy    = clip_norm[:, :, :2].reshape(T, -1)      # 104
    score = clip_norm[:, :,  2].reshape(T, -1)      # 52
    vxy   = np.zeros_like(xy)
    vxy[1:] = xy[1:] - xy[:-1]                      # 104
    ang   = np.stack([compute_angles(clip_norm[t]) for t in range(T)])  # 12
    feat  = np.concatenate([xy, vxy, score, ang], axis=1).astype(np.float32)
    assert feat.shape == (T, FEAT_DIM), f"Got {feat.shape}, expected (T, {FEAT_DIM})"
    return feat

print(f"Normalization + feature helpers ready  (FEAT_DIM={FEAT_DIM})")


In [ ]:
def _rotate_chain_2d(clip, vertex_idx, downstream_ids, angle_deg):
    """
    Rotate `downstream_ids` landmarks around `vertex_idx` by `angle_deg` (in XY)
    consistently across all frames. score channel untouched.
    Mutates and returns `clip` — shape (T, 52, 3).
    """
    theta = np.deg2rad(angle_deg)
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    for t in range(len(clip)):
        vx, vy = clip[t, vertex_idx, 0], clip[t, vertex_idx, 1]
        for lm_id in downstream_ids:
            x, y = clip[t, lm_id, 0], clip[t, lm_id, 1]
            dx, dy = x - vx, y - vy
            clip[t, lm_id, 0] = vx + cos_t * dx - sin_t * dy
            clip[t, lm_id, 1] = vy + sin_t * dx + cos_t * dy
    return clip

def corrupt_clip(clip_norm, rng):
    """
    Produce a synthetically-incorrect twin of a clean normalized (T, 52, 3) clip.
    Returns (corrupted_clip, feedback_vec) where feedback_vec is a (6,) array
    of applied corruption magnitude in degrees (zero for uncorrupted joints).
    """
    out = clip_norm.copy()
    n_corrupt = rng.integers(N_CORRUPT_JOINTS_MIN, N_CORRUPT_JOINTS_MAX + 1)
    joints = rng.choice(N_JOINTS, size=n_corrupt, replace=False)

    feedback = np.zeros(N_JOINTS, dtype=np.float32)
    for j in joints:
        name  = JOINT_NAMES[j]
        mag   = rng.uniform(CORRUPT_ANGLE_MIN_DEG, CORRUPT_ANGLE_MAX_DEG)
        sign  = rng.choice([-1.0, 1.0])
        angle = sign * mag
        _rotate_chain_2d(out, JOINT_VERTEX[name], CHAIN_DOWNSTREAM[name], angle)
        feedback[j] = mag  # unsigned magnitude (matches BlazePose Stage 2/3/4 semantics)

    # Global 2D jitter on all landmarks after rotation (score untouched)
    jitter = rng.normal(0.0, JITTER_SIGMA, size=out[..., :2].shape).astype(np.float32)
    out[..., :2] += jitter
    return out, feedback

def clean_feedback():
    return np.zeros(N_JOINTS, dtype=np.float32)

print("Corruption helpers ready")


In [ ]:
all_X, all_q, all_p, all_fb, all_meta = [], [], [], [], []
skipped_bad_pose = skipped_no_windows = 0

for _, row in meta.iterrows():
    pose_name = row["pose"]
    if pose_name not in POSE_TO_ID:
        skipped_bad_pose += 1
        continue
    pose_id = POSE_TO_ID[pose_name]

    clip_raw  = np.load(row["clip_path"])                  # (T, 52, 3)
    clip_norm = normalize_clip(clip_raw)

    feat_clean = clip_to_features(clip_norm)
    clip_corrupt, fb_vec = corrupt_clip(clip_norm, RNG)
    feat_corrupt = clip_to_features(clip_corrupt)
    assert feat_clean.shape == feat_corrupt.shape

    T = feat_clean.shape[0]
    starts = list(range(0, T - WINDOW_SIZE + 1, STRIDE))
    if not starts:
        skipped_no_windows += 1
        continue

    # Detection mask from score block (columns 208:260 in the 272-dim vector)
    detected = (feat_clean[:, SCORE_SLICE].max(axis=1) > VIS_THRESH)    # (T,)

    for start in starts:
        end = start + WINDOW_SIZE
        if detected[start:end].mean() < MIN_VIS_FRAC:
            continue

        # Clean twin
        all_X.append(feat_clean[start:end])
        all_q.append(LABEL_CORRECT)
        all_p.append(pose_id)
        all_fb.append(clean_feedback())
        all_meta.append({
            "clip_index":    int(row["clip_index"]),
            "pose":          pose_name,
            "quality_label": "correct",
            "window_start":  start,
            "source_file":   row["source_file"],
            "dataset":       "yognet_synthpose",
        })

        # Corrupted twin — same parent clip, same window start, opposite quality
        all_X.append(feat_corrupt[start:end])
        all_q.append(LABEL_INCORRECT)
        all_p.append(pose_id)
        all_fb.append(fb_vec)
        all_meta.append({
            "clip_index":    int(row["clip_index"]),
            "pose":          pose_name,
            "quality_label": "incorrect",
            "window_start":  start,
            "source_file":   row["source_file"],
            "dataset":       "yognet_synthpose",
        })

print(f"Clips skipped (bad pose)  : {skipped_bad_pose}")
print(f"Clips skipped (no window) : {skipped_no_windows}")
print(f"Total windows: {len(all_X)}  "
      f"(correct={sum(q==0 for q in all_q)}, incorrect={sum(q==1 for q in all_q)})")


In [ ]:
from sklearn.model_selection import train_test_split

meta_df = pd.DataFrame(all_meta)
clip_ids = meta_df["clip_index"].unique()
# Stratify clip-level split by the clip's pose so each split gets all 7 classes
clip_pose = {cid: meta_df[meta_df["clip_index"] == cid]["pose"].iloc[0]
             for cid in clip_ids}
clip_pose_labels = [POSE_TO_ID[clip_pose[c]] for c in clip_ids]

train_cids, temp_cids, _, temp_lbls = train_test_split(
    clip_ids, clip_pose_labels, test_size=0.30, random_state=42,
    stratify=clip_pose_labels)
val_cids, test_cids = train_test_split(
    temp_cids, test_size=0.50, random_state=42, stratify=temp_lbls)

train_set, val_set, test_set = set(train_cids), set(val_cids), set(test_cids)

def split_of(cid):
    if cid in train_set: return "train"
    if cid in val_set:   return "val"
    return "test"

X_train, X_val, X_test = [], [], []
q_train, q_val, q_test = [], [], []
p_train, p_val, p_test = [], [], []
fb_train, fb_val, fb_test = [], [], []
mt_train, mt_val, mt_test = [], [], []

for X, q, p, fb, m in zip(all_X, all_q, all_p, all_fb, all_meta):
    bucket = split_of(m["clip_index"])
    if bucket == "train":
        X_train.append(X); q_train.append(q); p_train.append(p)
        fb_train.append(fb); mt_train.append(m)
    elif bucket == "val":
        X_val.append(X); q_val.append(q); p_val.append(p)
        fb_val.append(fb); mt_val.append(m)
    else:
        X_test.append(X); q_test.append(q); p_test.append(p)
        fb_test.append(fb); mt_test.append(m)

X_train = np.stack(X_train).astype(np.float32)
X_val   = np.stack(X_val).astype(np.float32)
X_test  = np.stack(X_test).astype(np.float32)
y_quality_train = np.array(q_train, dtype=np.int64)
y_quality_val   = np.array(q_val,   dtype=np.int64)
y_quality_test  = np.array(q_test,  dtype=np.int64)
y_pose_train    = np.array(p_train, dtype=np.int64)
y_pose_val      = np.array(p_val,   dtype=np.int64)
y_pose_test     = np.array(p_test,  dtype=np.int64)
y_fb_train = np.stack(fb_train).astype(np.float32)
y_fb_val   = np.stack(fb_val).astype(np.float32)
y_fb_test  = np.stack(fb_test).astype(np.float32)

# Feature stats — train only (clean + corrupted together, matching BlazePose Stage 2)
N, T, D   = X_train.shape
feat_mean = X_train.reshape(-1, D).mean(axis=0).astype(np.float32)
feat_std  = X_train.reshape(-1, D).std(axis=0).astype(np.float32) + 1e-6
np.savez(f"{OUT}/feature_stats.npz", mean=feat_mean, std=feat_std)

# Save arrays
np.save(f"{OUT}/X_train.npy", X_train)
np.save(f"{OUT}/X_val.npy",   X_val)
np.save(f"{OUT}/X_test.npy",  X_test)
np.save(f"{OUT}/y_quality_train.npy", y_quality_train)
np.save(f"{OUT}/y_quality_val.npy",   y_quality_val)
np.save(f"{OUT}/y_quality_test.npy",  y_quality_test)
np.save(f"{OUT}/y_pose_train.npy", y_pose_train)
np.save(f"{OUT}/y_pose_val.npy",   y_pose_val)
np.save(f"{OUT}/y_pose_test.npy",  y_pose_test)
np.save(f"{OUT}/y_feedback_train.npy", y_fb_train)
np.save(f"{OUT}/y_feedback_val.npy",   y_fb_val)
np.save(f"{OUT}/y_feedback_test.npy",  y_fb_test)
pd.DataFrame(mt_train).to_csv(f"{OUT}/meta_train.csv", index=False)
pd.DataFrame(mt_val).to_csv(  f"{OUT}/meta_val.csv",   index=False)
pd.DataFrame(mt_test).to_csv( f"{OUT}/meta_test.csv",  index=False)

# Summary
print("=" * 55)
print("Stage 2′ complete (YogNet + SynthPose)")
print(f"  Train: {X_train.shape}  | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"  NaN in X_train: {np.isnan(X_train).sum()}")
print(f"  Train quality balance: correct={(y_quality_train==0).sum()} "
      f"incorrect={(y_quality_train==1).sum()}")
print("  Train pose distribution:")
for pid, pname in enumerate(POSE_NAMES):
    print(f"    [{pid}] {pname:22s} {(y_pose_train==pid).sum()}")
print(f"  Feedback target range (train): "
      f"min={y_fb_train.min():.2f}, max={y_fb_train.max():.2f}, "
      f"mean_nonzero={y_fb_train[y_fb_train>0].mean():.2f} deg")
print(f"  Feature stats saved: {OUT}/feature_stats.npz")


In [ ]:
# Stage 3' - TCN training on YogNet SynthPose features; saves best weights + checkpoint bundle.
import numpy as np
import pandas as pd
import os, torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report

OUT = "path/to/working/windows_synthpose"

X_train = np.load(f"{OUT}/X_train.npy")
X_val   = np.load(f"{OUT}/X_val.npy")
X_test  = np.load(f"{OUT}/X_test.npy")

y_q_train = np.load(f"{OUT}/y_quality_train.npy")
y_q_val   = np.load(f"{OUT}/y_quality_val.npy")
y_q_test  = np.load(f"{OUT}/y_quality_test.npy")
y_p_train = np.load(f"{OUT}/y_pose_train.npy")
y_p_val   = np.load(f"{OUT}/y_pose_val.npy")
y_p_test  = np.load(f"{OUT}/y_pose_test.npy")
y_fb_train = np.load(f"{OUT}/y_feedback_train.npy")
y_fb_val   = np.load(f"{OUT}/y_feedback_val.npy")
y_fb_test  = np.load(f"{OUT}/y_feedback_test.npy")

stats     = np.load(f"{OUT}/feature_stats.npz")
FEAT_MEAN = stats["mean"]
FEAT_STD  = stats["std"]

# Labels and names — identical to BlazePose Stage 3 for fair comparison
POSE_NAMES = ["cobra", "corpse", "downward_facing_dog",
              "mountain", "tree", "triangle", "warrior"]
N_POSES    = 7
N_JOINTS   = 6
JOINT_NAMES = ["left_elbow", "right_elbow", "left_knee",
               "right_knee", "left_hip",    "right_hip"]

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"Quality balance train: correct={(y_q_train==0).sum()} "
      f"incorrect={(y_q_train==1).sum()}")
print(f"FEAT_DIM (SynthPose): {X_train.shape[2]}")


In [ ]:
class YogaWindowDataset(Dataset):
    def __init__(self, X, y_q, y_p, y_fb, mean, std):
        X_std = (X - mean[None, None, :]) / std[None, None, :]
        self.X    = torch.from_numpy(X_std).float()
        self.y_q  = torch.from_numpy(y_q).long()
        self.y_p  = torch.from_numpy(y_p).long()
        self.y_fb = torch.from_numpy(y_fb).float()

    def __len__(self):  return len(self.y_q)
    def __getitem__(self, i):
        return self.X[i], self.y_q[i], self.y_p[i], self.y_fb[i]

train_ds = YogaWindowDataset(X_train, y_q_train, y_p_train, y_fb_train,
                             FEAT_MEAN, FEAT_STD)
val_ds   = YogaWindowDataset(X_val,   y_q_val,   y_p_val,   y_fb_val,
                             FEAT_MEAN, FEAT_STD)
test_ds  = YogaWindowDataset(X_test,  y_q_test,  y_p_test,  y_fb_test,
                             FEAT_MEAN, FEAT_STD)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False,
                          num_workers=2, pin_memory=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
class TCNBlock(nn.Module):
    """Causal TCN — left-pad only so output length == input length."""
    def __init__(self, cin, cout, k, dilation, pdrop=0.2):
        super().__init__()
        pad = (k - 1) * dilation
        self.pad1  = nn.ConstantPad1d((pad, 0), 0.0)
        self.conv1 = nn.Conv1d(cin, cout, k, padding=0, dilation=dilation)
        self.pad2  = nn.ConstantPad1d((pad, 0), 0.0)
        self.conv2 = nn.Conv1d(cout, cout, k, padding=0, dilation=dilation)
        self.drop  = nn.Dropout(pdrop)
        self.res   = nn.Conv1d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x):
        y = F.relu(self.conv1(self.pad1(x)))
        y = self.drop(y)
        y = F.relu(self.conv2(self.pad2(y)))
        y = self.drop(y)
        return y + self.res(x)

class YogaTCN(nn.Module):
    def __init__(self, feat_dim=272, hidden=128, layers=6,
                 n_joints=6, n_poses=7):
        super().__init__()
        blocks, cin = [], feat_dim
        for i in range(layers):
            blocks.append(TCNBlock(cin, hidden, k=3, dilation=2**i))
            cin = hidden
        self.tcn  = nn.Sequential(*blocks)
        self.pool = nn.AdaptiveAvgPool1d(1)

        self.quality_head  = nn.Linear(hidden, 2)
        self.pose_head     = nn.Linear(hidden, n_poses)
        self.feedback_head = nn.Linear(hidden, n_joints)

        # Soft conditioning: initialized to zeros so it has no effect at start
        self.cond_matrix = nn.Parameter(torch.zeros(n_poses, n_joints))

    def forward(self, x):
        h = self.tcn(x.transpose(1, 2))                  # (B, C, T)
        g = self.pool(h).squeeze(-1)                     # (B, hidden)
        quality_logits = self.quality_head(g)            # (B, 2)
        pose_logits    = self.pose_head(g)               # (B, 7)
        feedback_raw   = self.feedback_head(g)           # (B, 6)
        pose_probs     = F.softmax(pose_logits, dim=1)
        cond           = pose_probs @ self.cond_matrix
        feedback       = F.relu(feedback_raw + cond)     # non-negative degrees
        return quality_logits, pose_logits, feedback

FEAT_DIM = X_train.shape[2]    # 272 for SynthPose
model = YogaTCN(feat_dim=FEAT_DIM, hidden=128, layers=6,
                n_joints=N_JOINTS, n_poses=N_POSES).to(device)

total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTrainable parameters: {total:,}")


In [ ]:
QUALITY_W = 1.0
POSE_W    = 0.5
FB_W      = 0.4

criterion_q = nn.CrossEntropyLoss()

# Per-pose inverse-frequency weights (ref only — disabled by default, matching BlazePose Stage 3)
pose_counts  = np.bincount(y_p_train, minlength=N_POSES).astype(np.float32)
pose_weights = pose_counts.mean() / pose_counts
pose_weights_t = torch.from_numpy(pose_weights).to(device)
print("Per-pose inverse-frequency weights (ref only — disabled by default):")
for pid, pname in enumerate(POSE_NAMES):
    print(f"  [{pid}] {pname:22s} count={int(pose_counts[pid]):5d} "
          f"weight={pose_weights[pid]:.3f}")

criterion_p = nn.CrossEntropyLoss()
# Feedback loss: smooth-L1 applied functionally in the training loop

optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=5, factor=0.5
)
print("Loss, optimizer, scheduler ready")


In [ ]:
CHECKPOINT = "path/to/working/yoga_tcn_synthpose_best.pt"
MAX_EPOCHS = 80
PATIENCE   = 10

def eval_quality_acc(loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for Xb, yb_q, _, _ in loader:
            Xb, yb_q = Xb.to(device), yb_q.to(device)
            logits_q, _, _ = model(Xb)
            correct += (logits_q.argmax(1) == yb_q).sum().item()
            total   += yb_q.size(0)
    return correct / total

best_val_acc = 0.0
bad_epochs   = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    n_seen = 0

    for Xb, yb_q, yb_p, yb_fb in train_loader:
        Xb    = Xb.to(device, non_blocking=True)
        yb_q  = yb_q.to(device, non_blocking=True)
        yb_p  = yb_p.to(device, non_blocking=True)
        yb_fb = yb_fb.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits_q, logits_p, feedback = model(Xb)

        loss_q  = criterion_q(logits_q, yb_q)
        loss_p  = criterion_p(logits_p, yb_p)
        loss_fb = F.smooth_l1_loss(feedback, yb_fb)
        loss    = QUALITY_W * loss_q + POSE_W * loss_p + FB_W * loss_fb

        loss.backward()
        optimizer.step()
        running_loss += loss.item() * yb_q.size(0)
        n_seen       += yb_q.size(0)

    train_loss = running_loss / n_seen
    val_acc    = eval_quality_acc(val_loader)
    scheduler.step(val_acc)

    print(f"Epoch {epoch:03d} | loss={train_loss:.4f} | val_quality_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        bad_epochs   = 0
        torch.save(model.state_dict(), CHECKPOINT)
        print("  ✅ saved best checkpoint")
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print("  🛑 early stopping")
            break

print(f"\nBest val quality accuracy: {best_val_acc:.4f}")


In [ ]:
model.load_state_dict(torch.load(CHECKPOINT, map_location=device))
model.eval()

all_q_preds, all_q_true   = [], []
all_p_preds, all_p_true   = [], []
all_fb_preds, all_fb_true = [], []

with torch.no_grad():
    for Xb, yb_q, yb_p, yb_fb in test_loader:
        Xb = Xb.to(device)
        logits_q, logits_p, feedback = model(Xb)
        all_q_preds.append(logits_q.argmax(1).cpu().numpy())
        all_q_true.append(yb_q.numpy())
        all_p_preds.append(logits_p.argmax(1).cpu().numpy())
        all_p_true.append(yb_p.numpy())
        all_fb_preds.append(feedback.cpu().numpy())
        all_fb_true.append(yb_fb.numpy())

all_q_preds  = np.concatenate(all_q_preds);  all_q_true = np.concatenate(all_q_true)
all_p_preds  = np.concatenate(all_p_preds);  all_p_true = np.concatenate(all_p_true)
all_fb_preds = np.concatenate(all_fb_preds); all_fb_true = np.concatenate(all_fb_true)

print("=" * 55)
print("QUALITY HEAD")
print(f"Test accuracy: {(all_q_preds == all_q_true).mean():.4f}")
print(confusion_matrix(all_q_true, all_q_preds))
print(classification_report(all_q_true, all_q_preds,
                            target_names=["correct", "incorrect"]))

print("=" * 55)
print("POSE HEAD (7 classes)")
print(f"Test accuracy: {(all_p_preds == all_p_true).mean():.4f}")
print(classification_report(all_p_true, all_p_preds,
                            target_names=POSE_NAMES, digits=3))

print("=" * 55)
print("FEEDBACK HEAD — per-joint MAE (degrees)")
for j, name in enumerate(JOINT_NAMES):
    mae = np.abs(all_fb_preds[:, j] - all_fb_true[:, j]).mean()
    print(f"  {name:20s} MAE={mae:5.2f} deg")

# ---- Pose-angle and pose-landmark reference means for Stage 7′ inference ----
# Feature layout (SynthPose):
#   xy      : columns   0..103    (52*2)
#   vxy     : columns 104..207    (52*2)
#   score   : columns 208..259    (52)
#   angles  : columns 260..271    (12; first 6 are the joint angles)
ANGLE_START = 260
ANGLE_COLS  = 6
XY_START    = 0
XY_COLS     = 104                 # 52 keypoints × 2

clean_mask = (y_q_train == 0)
pose_angle_means    = {}
pose_landmark_means = {}
for pid in range(N_POSES):
    m = (y_p_train == pid) & clean_mask
    if m.sum() > 0:
        angles = X_train[m, :, ANGLE_START:ANGLE_START + ANGLE_COLS].mean(axis=(0, 1))
        pose_angle_means[pid] = angles.astype(np.float32)
        xy = X_train[m, :, XY_START:XY_START + XY_COLS].mean(axis=(0, 1)).reshape(52, 2)
        pose_landmark_means[pid] = xy.astype(np.float32)
    else:
        pose_angle_means[pid]    = np.zeros(ANGLE_COLS,  dtype=np.float32)
        pose_landmark_means[pid] = np.zeros((52, 2),     dtype=np.float32)

# ---- Save checkpoint bundle for Stages 5′/7′ / ablation ----
torch.save({
    "model":               model.state_dict(),
    "feat_dim":            int(FEAT_DIM),
    "hidden":              128,
    "layers":              6,
    "n_joints":            N_JOINTS,
    "n_poses":             N_POSES,
    "n_keypoints":         52,                    # topology marker (vs 33 for BlazePose)
    "topology":            "synthpose-52",
    "joint_names":         JOINT_NAMES,
    "pose_names":          POSE_NAMES,
    "pose_to_id":          {p: i for i, p in enumerate(POSE_NAMES)},
    "feat_mean":           FEAT_MEAN.tolist(),
    "feat_std":            FEAT_STD.tolist(),
    "pose_angle_means":    {k: v.tolist() for k, v in pose_angle_means.items()},
    "pose_landmark_means": {k: v.tolist() for k, v in pose_landmark_means.items()},
    "max_feedback_deg":    45.0,
    "angle_start":         ANGLE_START,
    "angle_cols":          ANGLE_COLS,
    "xy_start":            XY_START,
    "xy_cols":             XY_COLS,
    "window_size":         int(X_train.shape[1]),
}, "path/to/working/yoga_tcn_synthpose.pt")

print("\n✅ SynthPose checkpoint saved: path/to/working/yoga_tcn_synthpose.pt")
print("   Ready for Stage 5′ (YogNet SynthPose test eval) and Stage 8 (ablation)")


In [ ]:
# Diagnostic: break down feedback-head MAE by sample type (matches BlazePose Stage 6 metric).
import numpy as np

corrupted_mask = (all_q_true == 1)
correct_mask   = (all_q_true == 0)
print(f"Test samples: {len(all_q_true)}  (corrupted={corrupted_mask.sum()}, correct={correct_mask.sum()})")

print("\n" + "=" * 55)
print("FEEDBACK HEAD — MAE breakdown (deg)")
print(f"  {'joint':20s}  {'corrupted':>10s}  {'correct':>10s}  {'all':>10s}")
for j, name in enumerate(JOINT_NAMES):
    mae_corr = np.abs(all_fb_preds[corrupted_mask, j] - all_fb_true[corrupted_mask, j]).mean()
    mae_clean = np.abs(all_fb_preds[correct_mask, j]   - all_fb_true[correct_mask, j]).mean()
    mae_all   = np.abs(all_fb_preds[:, j]              - all_fb_true[:, j]).mean()
    print(f"  {name:20s}  {mae_corr:10.2f}  {mae_clean:10.2f}  {mae_all:10.2f}")

# Compare directly to BlazePose Stage 6 numbers you printed before
bp_stage6_corrupted = {
    "left_elbow": 5.44, "right_elbow": 7.68, "left_knee": 4.81,
    "right_knee": 7.55, "left_hip": 5.71, "right_hip": 5.28,
}
print("\n" + "=" * 55)
print("Side-by-side: SynthPose (corrupted-only) vs BlazePose Stage 6 (corrupted-only)")
print(f"  {'joint':20s}  {'SynthPose':>10s}  {'BlazePose':>10s}  {'delta':>10s}")
for j, name in enumerate(JOINT_NAMES):
    sp = np.abs(all_fb_preds[corrupted_mask, j] - all_fb_true[corrupted_mask, j]).mean()
    bp = bp_stage6_corrupted[name]
    delta = sp - bp
    marker = "↓" if delta < 0 else "↑"
    print(f"  {name:20s}  {sp:10.2f}  {bp:10.2f}  {delta:+8.2f}{marker}")


In [ ]:
# Stage 8 - Ablation: BlazePose vs SynthPose. BlazePose numbers are hardcoded from cs7150-video2; SynthPose numbers from Stage 3' eval.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.metrics import confusion_matrix

RESULTS_DIR = "path/to/working/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

POSE_NAMES = ["cobra", "corpse", "downward_facing_dog",
              "mountain", "tree", "triangle", "warrior"]
JOINT_NAMES = ["left_elbow", "right_elbow", "left_knee",
               "right_knee", "left_hip",    "right_hip"]

# Existing palette (from cs7150-video2 viz cells)
C_BLAZEPOSE = "#3b6aad"   # cobalt — used for Stage 3 pretrain / BlazePose
C_SYNTHPOSE = "#2e8b57"   # sea green — used for Stage 6 finetune / SynthPose

# ======== BlazePose numbers (Stage 6, YogNet test) — from cs7150-video2 ========
# Cell 59 quality confusion
bp_quality_cm = np.array([[426, 62],
                          [ 21, 467]])
# Cell 57 Stage 6 per-pose F1
bp_pose_f1 = {"cobra": 0.975, "corpse": 0.922, "downward_facing_dog": 0.966,
              "mountain": 0.970, "tree": 0.988, "triangle": 0.953, "warrior": 0.959}
# Cell 58 Stage 6 per-joint MAE on corrupted samples (degrees)
bp_feedback_mae = {"left_elbow": 5.44, "right_elbow": 7.68,
                   "left_knee":  4.81, "right_knee":  7.55,
                   "left_hip":   5.71, "right_hip":   5.28}

# ======== SynthPose numbers (Stage 3′, YogNet test) — from the all_* arrays ========
# These arrays are still in memory from Stage 3′-6 evaluation.
sp_quality_cm = confusion_matrix(all_q_true, all_q_preds)

# Per-pose F1 from the sklearn classification_report would work, but we can
# compute directly from predictions to avoid a dependency
from sklearn.metrics import f1_score
sp_pose_f1_arr = f1_score(all_p_true, all_p_preds, average=None,
                          labels=list(range(len(POSE_NAMES))))
sp_pose_f1 = {name: float(sp_pose_f1_arr[i]) for i, name in enumerate(POSE_NAMES)}

# Per-joint MAE on corrupted samples only (matches BlazePose's metric exactly)
corrupted_mask = (all_q_true == 1)
sp_feedback_mae = {}
for j, name in enumerate(JOINT_NAMES):
    sp_feedback_mae[name] = float(
        np.abs(all_fb_preds[corrupted_mask, j] - all_fb_true[corrupted_mask, j]).mean()
    )

print("Loaded comparison data:")
print(f"  BlazePose quality acc: {bp_quality_cm.diagonal().sum() / bp_quality_cm.sum():.4f}")
print(f"  SynthPose quality acc: {sp_quality_cm.diagonal().sum() / sp_quality_cm.sum():.4f}")
print(f"  BlazePose mean pose F1: {np.mean(list(bp_pose_f1.values())):.4f}")
print(f"  SynthPose mean pose F1: {np.mean(list(sp_pose_f1.values())):.4f}")
print(f"  BlazePose mean feedback MAE: {np.mean(list(bp_feedback_mae.values())):.2f}°")
print(f"  SynthPose mean feedback MAE: {np.mean(list(sp_feedback_mae.values())):.2f}°")


In [ ]:
# Figure 1 - quality-head confusion, BlazePose vs SynthPose side by side.
labels = ["correct", "incorrect"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, cm, title in [
    (axes[0], bp_quality_cm, "BlazePose Stage 6 (cs7150-video2)"),
    (axes[1], sp_quality_cm, "SynthPose Stage 3′ (this notebook)"),
]:
    im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=cm.max())

    # Cell annotations — same format as Cell 59
    for i in range(2):
        for j in range(2):
            value = cm[i, j]
            pct = value / cm.sum() * 100
            text_color = "white" if value > cm.max() * 0.5 else "black"
            ax.text(j, i, f"{value}\n({pct:.1f}%)",
                    ha="center", va="center", color=text_color,
                    fontsize=12, fontweight="bold")

    ax.set_xticks([0, 1]);  ax.set_yticks([0, 1])
    ax.set_xticklabels(labels);  ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    acc = cm.diagonal().sum() / cm.sum()
    ax.set_title(f"{title}\nquality acc = {acc*100:.1f}%")

    # Recall sidebar — placed outside right edge of each subplot, monospace for alignment
    recalls = [cm[i, i] / cm[i].sum() for i in range(2)]
    ax.text(1.01, 0.5,
            f"recall\ncorrect   {recalls[0]:.1%}\nincorrect {recalls[1]:.1%}",
            transform=ax.transAxes, ha="left", va="center",
            fontsize=9, color="#555", family="monospace")

plt.tight_layout(w_pad=3.0)
plt.savefig(f"{RESULTS_DIR}/viz_ablation_quality_confusion.png",
            dpi=120, bbox_inches="tight")
plt.show()
print("Saved: viz_ablation_quality_confusion.png")


In [ ]:
# Figure 2 - per-pose F1, BlazePose vs SynthPose (paired bars).
x = np.arange(len(POSE_NAMES))
width = 0.38  # matches Cell 58 paired-bar width

fig, ax = plt.subplots(figsize=(12, 5))

ax.bar(x - width/2, [bp_pose_f1[p] for p in POSE_NAMES], width,
       label="BlazePose — Stage 6 (YogNet finetune)", color=C_BLAZEPOSE)
ax.bar(x + width/2, [sp_pose_f1[p] for p in POSE_NAMES], width,
       label="SynthPose — Stage 3′ (YogNet only)", color=C_SYNTHPOSE)

# Bar-value annotations — same format as Cell 58
for i, p in enumerate(POSE_NAMES):
    ax.text(i - width/2, bp_pose_f1[p] + 0.015,
            f"{bp_pose_f1[p]:.3f}", ha="center", fontsize=8)
    ax.text(i + width/2, sp_pose_f1[p] + 0.015,
            f"{sp_pose_f1[p]:.3f}", ha="center", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([p.replace("_", "\n") for p in POSE_NAMES], fontsize=9)
ax.set_ylabel("F1 score")
ax.set_ylim(0, 1.08)
ax.set_title("Per-pose F1 — BlazePose vs SynthPose (YogNet test)", fontsize=12)
ax.axhline(1.0, linestyle=":", color="gray", alpha=0.3)
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/viz_ablation_per_pose_f1.png",
            dpi=120, bbox_inches="tight")
plt.show()
print("Saved: viz_ablation_per_pose_f1.png")


In [ ]:
# Figure 3 - feedback-head per-joint MAE, BlazePose vs SynthPose (corrupted-only samples).
x = np.arange(len(JOINT_NAMES))
width = 0.38  # matches Cell 58

bp_vals = [bp_feedback_mae[j] for j in JOINT_NAMES]
sp_vals = [sp_feedback_mae[j] for j in JOINT_NAMES]

fig, ax = plt.subplots(figsize=(10, 4.5))

ax.bar(x - width/2, bp_vals, width,
       label="BlazePose — Stage 6 (corrupted samples)",
       color=C_BLAZEPOSE)
ax.bar(x + width/2, sp_vals, width,
       label="SynthPose — Stage 3′ (corrupted samples)",
       color=C_SYNTHPOSE)

# Annotate each bar with its value — same format as Cell 58
for i, v in enumerate(bp_vals):
    ax.text(i - width/2, v + 0.15, f"{v:.2f}", ha="center", fontsize=9)
for i, v in enumerate(sp_vals):
    ax.text(i + width/2, v + 0.15, f"{v:.2f}", ha="center", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels([j.replace("_", "\n") for j in JOINT_NAMES], fontsize=9)
ax.set_ylabel("Feedback head MAE (degrees)")
ax.set_ylim(0, max(max(bp_vals), max(sp_vals)) * 1.2)
ax.set_title("Feedback head per-joint MAE — BlazePose vs SynthPose",
             fontsize=12)
ax.legend(fontsize=9, loc="upper left")
ax.grid(alpha=0.3, axis="y")

# Summary annotation — mirrors Cell 58's annotation callout style
n_better  = sum(1 for j in JOINT_NAMES if sp_feedback_mae[j] < bp_feedback_mae[j])
n_worse   = len(JOINT_NAMES) - n_better
best_j = min(JOINT_NAMES, key=lambda j: sp_feedback_mae[j] - bp_feedback_mae[j])
best_delta = sp_feedback_mae[best_j] - bp_feedback_mae[best_j]
ax.text(0.98, 0.95,
        f"SynthPose better on {n_better}/6 joints\n"
        f"biggest gain: {best_j} ({best_delta:+.2f}°)\n"
        f"knees benefit from medial+lateral markers",
        transform=ax.transAxes, fontsize=9, ha="right", va="top",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#fff2cc",
                  edgecolor="#c5a900", alpha=0.9))

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/viz_ablation_feedback_mae.png",
            dpi=120, bbox_inches="tight")
plt.show()
print("Saved: viz_ablation_feedback_mae.png")


In [ ]:
# Stage 7' - real-time inference with SynthPose keypoints + skeleton overlay.
import os, cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from transformers import (
    AutoProcessor,
    RTDetrForObjectDetection,
    VitPoseForPoseEstimation,
)

CKPT_PATH = "path/to/working/yoga_tcn_synthpose.pt"
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)

FEAT_DIM      = ckpt["feat_dim"]       # 272
HIDDEN        = ckpt["hidden"]
LAYERS        = ckpt["layers"]
N_JOINTS      = ckpt["n_joints"]       # 6
N_POSES       = ckpt["n_poses"]        # 7
N_KEYPOINTS   = ckpt["n_keypoints"]    # 52
JOINT_NAMES   = ckpt["joint_names"]
POSE_NAMES    = ckpt["pose_names"]
FEAT_MEAN     = np.array(ckpt["feat_mean"],  dtype=np.float32)
FEAT_STD      = np.array(ckpt["feat_std"],   dtype=np.float32)
WINDOW_SIZE   = ckpt["window_size"]
ANGLE_START   = ckpt["angle_start"]    # 260
ANGLE_COLS    = ckpt["angle_cols"]     # 6
XY_START      = ckpt["xy_start"]       # 0
XY_COLS       = ckpt["xy_cols"]        # 104
MAX_FB_DEG    = ckpt["max_feedback_deg"]

# Reshape POSE_LANDMARK_MEANS to (52, 2) instead of (33, 2) for BlazePose
POSE_ANGLE_MEANS    = {int(k): np.array(v, dtype=np.float32)
                       for k, v in ckpt["pose_angle_means"].items()}
POSE_LANDMARK_MEANS = {int(k): np.array(v, dtype=np.float32).reshape(N_KEYPOINTS, 2)
                       for k, v in ckpt["pose_landmark_means"].items()}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Checkpoint topology: {ckpt.get('topology', 'unknown')}")
print(f"Device: {device}")
print(f"Poses ({N_POSES}): {POSE_NAMES}")
print(f"Window size: {WINDOW_SIZE} | Feat dim: {FEAT_DIM} | Keypoints: {N_KEYPOINTS}")


In [ ]:
# TCN architecture — identical to Stage 3′, redefined here for inference standalone.
class TCNBlock(nn.Module):
    def __init__(self, cin, cout, k, dilation, pdrop=0.2):
        super().__init__()
        pad = (k - 1) * dilation
        self.pad1  = nn.ConstantPad1d((pad, 0), 0.0)
        self.conv1 = nn.Conv1d(cin, cout, k, padding=0, dilation=dilation)
        self.pad2  = nn.ConstantPad1d((pad, 0), 0.0)
        self.conv2 = nn.Conv1d(cout, cout, k, padding=0, dilation=dilation)
        self.drop  = nn.Dropout(pdrop)
        self.res   = nn.Conv1d(cin, cout, 1) if cin != cout else nn.Identity()
    def forward(self, x):
        y = F.relu(self.conv1(self.pad1(x)))
        y = self.drop(y)
        y = F.relu(self.conv2(self.pad2(y)))
        y = self.drop(y)
        return y + self.res(x)

class YogaTCN(nn.Module):
    def __init__(self, feat_dim, hidden, layers, n_joints, n_poses):
        super().__init__()
        blocks, cin = [], feat_dim
        for i in range(layers):
            blocks.append(TCNBlock(cin, hidden, k=3, dilation=2**i))
            cin = hidden
        self.tcn  = nn.Sequential(*blocks)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.quality_head  = nn.Linear(hidden, 2)
        self.pose_head     = nn.Linear(hidden, n_poses)
        self.feedback_head = nn.Linear(hidden, n_joints)
        self.cond_matrix   = nn.Parameter(torch.zeros(n_poses, n_joints))
    def forward(self, x):
        h = self.tcn(x.transpose(1, 2))
        g = self.pool(h).squeeze(-1)
        qs = self.quality_head(g)
        ps = self.pose_head(g)
        fr = self.feedback_head(g)
        pp = F.softmax(ps, dim=1)
        fb = F.relu(fr + pp @ self.cond_matrix)
        return qs, ps, fb

model_inf = YogaTCN(FEAT_DIM, HIDDEN, LAYERS, N_JOINTS, N_POSES)
model_inf.load_state_dict(ckpt["model"])
model_inf.to(device).eval()
print("TCN ready for inference")


In [ ]:
# Load RTDetr + SynthPose-ViTPose-base (same models as Stage 1′)
print("Loading RTDetr person detector...")
det_proc_inf  = AutoProcessor.from_pretrained("PekingU/rtdetr_r50vd_coco_o365")
det_model_inf = RTDetrForObjectDetection.from_pretrained(
    "PekingU/rtdetr_r50vd_coco_o365"
).to(device).eval()

print("Loading SynthPose ViTPose-base...")
pose_proc_inf  = AutoProcessor.from_pretrained("yonigozlan/synthpose-vitpose-base-hf")
pose_model_inf = VitPoseForPoseEstimation.from_pretrained(
    "yonigozlan/synthpose-vitpose-base-hf"
).to(device).eval()

# ---- SynthPose keypoint index references (mirror Stage 2′ but redefined for inference standalone) ----
# COCO-17 (0..16): nose, l_eye, r_eye, l_ear, r_ear, l_sho, r_sho,
#                  l_elb, r_elb, l_wri, r_wri, l_hip, r_hip, l_kne, r_kne, l_ank, r_ank
# Anatomical (17..51): sternum, rshoulder, lshoulder, r_lelbow, l_lelbow,
#   r_melbow, l_melbow, r_lwrist, l_lwrist, r_mwrist, l_mwrist,
#   r_ASIS, l_ASIS, r_PSIS, l_PSIS, r_knee, l_knee, r_mknee, l_mknee,
#   r_ankle, l_ankle, r_mankle, l_mankle, r_5meta, l_5meta,
#   r_toe, l_toe, r_big_toe, l_big_toe, l_calc, r_calc, C7, L2, T11, T6

# Keypoint aliases used in inference — MUST match Stage 2′'s KP namespace indices
I_LSHOULDER  = 19      # anatomical lshoulder
I_RSHOULDER  = 18      # anatomical rshoulder
I_L_ASIS     = 29      # left pelvis anterior
I_R_ASIS     = 28      # right pelvis anterior
I_LELBOW     = 7       # COCO left elbow
I_RELBOW     = 8       # COCO right elbow
I_LWRIST     = 9       # COCO left wrist
I_RWRIST     = 10      # COCO right wrist
I_L_KNEE     = 33      # anatomical left knee (lateral)
I_R_KNEE     = 32      # anatomical right knee (lateral)
I_L_ANKLE    = 37      # anatomical left ankle (lateral malleolus)
I_R_ANKLE    = 36      # anatomical right ankle

VIS_THRESH_INF = 0.3
DET_THRESHOLD_INF = 0.3

def detect_keypoints_inf(frame_bgr):
    """Run RTDetr + SynthPose on a BGR frame. Returns (52, 3) array of
    (x_norm, y_norm, score) where x/y are fractions of frame dims, or None
    if no person detected. Normalizes coordinates to match BlazePose convention."""
    H, W = frame_bgr.shape[:2]
    pil = Image.fromarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))

    # Stage 1: detect largest person
    inputs = det_proc_inf(images=pil, return_tensors="pt").to(device)
    with torch.no_grad():
        out = det_model_inf(**inputs)
    results = det_proc_inf.post_process_object_detection(
        out, target_sizes=torch.tensor([(H, W)]),
        threshold=DET_THRESHOLD_INF,
    )[0]
    person_mask  = results["labels"] == 0
    person_boxes = results["boxes"][person_mask].cpu().numpy()
    if len(person_boxes) == 0:
        return None
    x1, y1, x2, y2 = person_boxes.T
    areas = (x2 - x1) * (y2 - y1)
    i = int(np.argmax(areas))
    box_xywh = np.array([x1[i], y1[i], x2[i] - x1[i], y2[i] - y1[i]])

    # Stage 2: run pose on that person
    inputs = pose_proc_inf(pil, boxes=[[box_xywh]], return_tensors="pt").to(device)
    with torch.no_grad():
        out = pose_model_inf(**inputs)
    pose_results = pose_proc_inf.post_process_pose_estimation(out, boxes=[[box_xywh]])
    person = pose_results[0][0]
    kpts   = np.asarray(person["keypoints"], dtype=np.float32)   # (52, 2) in pixels
    scores = np.asarray(person["scores"],    dtype=np.float32)   # (52,)

    # Normalize pixel coords to (0,1) so downstream normalize_frame + drawing
    # can use the same multiply-by-W/H pattern as BlazePose
    kpts[:, 0] /= float(W)
    kpts[:, 1] /= float(H)
    return np.concatenate([kpts, scores[:, None]], axis=1)       # (52, 3)

def normalize_frame_inf(kp):
    """Hip-center (ASIS midpoint) + anatomical shoulder scale. Mirrors Stage 2′."""
    kp = kp.copy()
    hip = (kp[I_L_ASIS, :2] + kp[I_R_ASIS, :2]) / 2.0
    kp[:, :2] -= hip
    scale = np.linalg.norm(kp[I_LSHOULDER, :2] - kp[I_RSHOULDER, :2]) + 1e-6
    kp[:, :2] /= scale
    return kp

def _angle_3pts_2d_inf(a, b, c):
    ba = a - b; bc = c - b
    cos = np.clip(np.dot(ba, bc) /
                  (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8), -1, 1)
    return float(np.degrees(np.arccos(cos)))

ANGLE_TRIPLES_INF = {
    "left_elbow":  (I_LSHOULDER,  I_LELBOW,  I_LWRIST),
    "right_elbow": (I_RSHOULDER,  I_RELBOW,  I_RWRIST),
    "left_knee":   (I_L_ASIS,     I_L_KNEE,  I_L_ANKLE),
    "right_knee":  (I_R_ASIS,     I_R_KNEE,  I_R_ANKLE),
    "left_hip":    (I_LSHOULDER,  I_L_ASIS,  I_L_KNEE),
    "right_hip":   (I_RSHOULDER,  I_R_ASIS,  I_R_KNEE),
}

def compute_angles_inf(kp):
    """12-dim angle vec — 6 joint angles + 3 diffs + 3 means. kp: (52, 3)."""
    xy, score = kp[:, :2], kp[:, 2]
    def safe(name):
        a, b, c = ANGLE_TRIPLES_INF[name]
        if min(score[a], score[b], score[c]) >= VIS_THRESH_INF:
            return _angle_3pts_2d_inf(xy[a], xy[b], xy[c])
        return 0.0
    l_el = safe("left_elbow");  r_el = safe("right_elbow")
    l_kn = safe("left_knee");   r_kn = safe("right_knee")
    l_hp = safe("left_hip");    r_hp = safe("right_hip")
    return np.array([l_el, r_el, l_kn, r_kn, l_hp, r_hp,
                     l_el - r_el, l_kn - r_kn, l_hp - r_hp,
                     (l_el + r_el)/2, (l_kn + r_kn)/2, (l_hp + r_hp)/2],
                    dtype=np.float32)

def build_feature_window_inf(kp_buffer):
    """Mirrors Stage 2′ clip_to_features but standardizes at the end."""
    clip = np.stack(kp_buffer, axis=0)                   # (T, 52, 3)
    norm = np.stack([normalize_frame_inf(clip[t]) for t in range(len(clip))])
    T    = len(norm)
    xy   = norm[:, :, :2].reshape(T, -1)                 # 104
    sc   = norm[:, :,  2].reshape(T, -1)                 # 52
    vxy  = np.zeros_like(xy); vxy[1:] = xy[1:] - xy[:-1] # 104
    ang  = np.stack([compute_angles_inf(norm[t]) for t in range(T)])  # 12
    feat = np.concatenate([xy, vxy, sc, ang], axis=1).astype(np.float32)
    return (feat - FEAT_MEAN[None]) / FEAT_STD[None]

print("SynthPose detector + feature helpers ready")


In [ ]:
# Severity thresholds (degrees) — identical to BlazePose Stage 7
THRESH_GREEN  = 10.0
THRESH_YELLOW = 25.0
CONF_GATE     = 0.6

# BGR colors — identical to BlazePose Stage 7
COL_GREEN  = (50,  205,  50)
COL_YELLOW = (0,   215, 255)
COL_RED    = (0,    60, 220)
COL_GHOST  = (200, 200, 200)
COL_PANEL  = (20,   20,  20)

# Joint-vertex indices — pivots for the 6 feedback joints in SynthPose topology
JOINT_VERTEX_IDS = [I_LELBOW, I_RELBOW, I_L_KNEE, I_R_KNEE, I_L_ASIS, I_R_ASIS]

# Feedback-relevant landmarks rendered in the detected-skeleton overlay.
# Choose a clean readable subset of the 52 keypoints — shoulders, elbows,
# wrists, pelvis, knees, ankles.
FEEDBACK_LM_IDS = {
    I_LSHOULDER, I_RSHOULDER,
    I_LELBOW, I_RELBOW,
    I_LWRIST, I_RWRIST,
    I_L_ASIS, I_R_ASIS,
    I_L_KNEE, I_R_KNEE,
    I_L_ANKLE, I_R_ANKLE,
}

# Skeleton connections — SynthPose 52-keypoint topology. Mirrors BlazePose
# Stage 7's IDEAL_CONNECTIONS list structurally (arms + torso + legs).
IDEAL_CONNECTIONS = [
    # Arms: shoulder -> elbow -> wrist (L, R)
    (I_LSHOULDER, I_LELBOW), (I_LELBOW, I_LWRIST),
    (I_RSHOULDER, I_RELBOW), (I_RELBOW, I_RWRIST),
    # Torso: shoulders to ASIS (L, R) + pelvis crossbar
    (I_LSHOULDER, I_L_ASIS), (I_RSHOULDER, I_R_ASIS), (I_L_ASIS, I_R_ASIS),
    # Legs: ASIS -> knee -> ankle (L, R)
    (I_L_ASIS, I_L_KNEE), (I_L_KNEE, I_L_ANKLE),
    (I_R_ASIS, I_R_KNEE), (I_R_KNEE, I_R_ANKLE),
]

# Map from rendered landmark to its feedback-joint name for color routing
LM_TO_JOINT = {
    I_LSHOULDER: "left_hip",   I_RSHOULDER: "right_hip",   # torso shares hip joint
    I_LELBOW:    "left_elbow", I_RELBOW:    "right_elbow",
    I_LWRIST:    "left_elbow", I_RWRIST:    "right_elbow",
    I_L_ASIS:    "left_hip",   I_R_ASIS:    "right_hip",
    I_L_KNEE:    "left_knee",  I_R_KNEE:    "right_knee",
    I_L_ANKLE:   "left_knee",  I_R_ANKLE:   "right_knee",
}

def severity_color(deg):
    if deg < THRESH_GREEN:  return COL_GREEN
    if deg < THRESH_YELLOW: return COL_YELLOW
    return COL_RED

def bone_color(lm_a, lm_b, sev):
    da = sev.get(LM_TO_JOINT.get(lm_a, ""), 0.0)
    db = sev.get(LM_TO_JOINT.get(lm_b, ""), 0.0)
    return severity_color(max(da, db))

print("Rendering constants ready")


In [ ]:
def get_ideal_pixel_coords(kp_raw, pose_id, frame_h, frame_w):
    """Project the stored pose-mean landmark geometry onto the current frame,
    anchored at the user's ASIS-midpoint and scaled by shoulder width."""
    ref = POSE_LANDMARK_MEANS.get(pose_id)
    if ref is None:
        return None
    hip_px = ((kp_raw[I_L_ASIS, 0] + kp_raw[I_R_ASIS, 0]) / 2) * frame_w
    hip_py = ((kp_raw[I_L_ASIS, 1] + kp_raw[I_R_ASIS, 1]) / 2) * frame_h
    sh_dx  = (kp_raw[I_LSHOULDER, 0] - kp_raw[I_RSHOULDER, 0]) * frame_w
    sh_dy  = (kp_raw[I_LSHOULDER, 1] - kp_raw[I_RSHOULDER, 1]) * frame_h
    scale  = np.sqrt(sh_dx**2 + sh_dy**2) + 1e-6
    return {lm_id: (int(hip_px + ref[lm_id, 0] * scale),
                    int(hip_py + ref[lm_id, 1] * scale))
            for lm_id in FEEDBACK_LM_IDS}

def draw_detected_skeleton(frame, kp_raw, sev, H, W):
    # Bones
    for a, b in IDEAL_CONNECTIONS:
        xa = int(kp_raw[a, 0] * W); ya = int(kp_raw[a, 1] * H)
        xb = int(kp_raw[b, 0] * W); yb = int(kp_raw[b, 1] * H)
        cv2.line(frame, (xa, ya), (xb, yb), bone_color(a, b, sev), 2)
    # Joints
    vertex_set = set(JOINT_VERTEX_IDS)
    for lm_id in FEEDBACK_LM_IDS:
        px = int(kp_raw[lm_id, 0] * W); py = int(kp_raw[lm_id, 1] * H)
        joint_name = LM_TO_JOINT.get(lm_id, "")
        col = severity_color(sev.get(joint_name, 0.0))
        r = 7 if lm_id in vertex_set else 4
        cv2.circle(frame, (px, py), r, col, -1)

def draw_ideal_skeleton(frame, kp_raw, pose_id, pose_conf, H, W):
    coords = get_ideal_pixel_coords(kp_raw, pose_id, H, W)
    if coords is None:
        return
    overlay = frame.copy()
    for a, b in IDEAL_CONNECTIONS:
        if a in coords and b in coords:
            cv2.line(overlay, coords[a], coords[b], COL_GHOST, 2)
    vertex_set = set(JOINT_VERTEX_IDS)
    for lm_id in FEEDBACK_LM_IDS:
        if lm_id in coords:
            r = 6 if lm_id in vertex_set else 3
            cv2.circle(overlay, coords[lm_id], r, COL_GHOST, -1)
    cv2.addWeighted(overlay, pose_conf, frame, 1.0 - pose_conf, 0, frame)

def draw_text_panel(frame, result):
    sev = result["severities"]
    lines = [
        f"Pose : {result['pose_name']} ({result['pose_conf']:.0%})",
        f"Quality: {result['quality_score']:.2f}  (1=ideal)",
        f"Mean dev: {result['mean_dev']:.1f} deg",
    ]
    for name in sorted(sev, key=lambda k: -sev[k]):
        deg = sev[name]
        if deg >= THRESH_GREEN:
            flag = "!!" if deg >= THRESH_YELLOW else " !"
            lines.append(f"  {flag} {name}: {deg:.1f}\u00b0")

    panel_h = len(lines) * 22 + 12
    cv2.rectangle(frame, (5, 5), (290, panel_h), COL_PANEL, -1)
    cv2.rectangle(frame, (5, 5), (290, panel_h), (80, 80, 80), 1)
    for i, line in enumerate(lines):
        cv2.putText(frame, line, (10, 24 + i * 22),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.52, (240, 240, 240), 1, cv2.LINE_AA)

print("Draw functions ready")


In [ ]:
INFER_STRIDE = 8   # run TCN every N frames

def run_yoga_inference(video_in, video_out):
    cap = cv2.VideoCapture(video_in)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open {video_in}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    W   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    writer = cv2.VideoWriter(video_out, cv2.VideoWriter_fourcc(*"mp4v"),
                             fps, (W, H))

    kp_buffer   = []
    last_result = None
    last_kp_raw = None
    frame_idx   = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        kp_raw         = detect_keypoints_inf(frame)
        detected_frame = kp_raw is not None

        kp_buffer.append(kp_raw if detected_frame
                         else np.zeros((N_KEYPOINTS, 3), dtype=np.float32))
        if len(kp_buffer) > WINDOW_SIZE:
            kp_buffer.pop(0)
        if detected_frame:
            last_kp_raw = kp_raw

        # Run TCN every INFER_STRIDE frames once buffer is full
        if len(kp_buffer) == WINDOW_SIZE and frame_idx % INFER_STRIDE == 0:
            feat = build_feature_window_inf(kp_buffer)
            xt = torch.from_numpy(feat[None]).to(device)
            with torch.no_grad():
                logits_q, logits_p, feedback = model_inf(xt)
            q_probs = torch.softmax(logits_q, dim=1).cpu().numpy()[0]
            p_probs = torch.softmax(logits_p, dim=1).cpu().numpy()[0]
            fb_deg  = feedback.cpu().numpy()[0]
            pose_id = int(np.argmax(p_probs))
            last_result = {
                "pose_id":       pose_id,
                "pose_name":     POSE_NAMES[pose_id],
                "pose_conf":     float(p_probs[pose_id]),
                "quality_score": float(q_probs[0]),
                "mean_dev":      float(fb_deg.mean()),
                "severities":    {JOINT_NAMES[j]: float(fb_deg[j])
                                  for j in range(N_JOINTS)},
            }

        render = frame.copy()
        if last_result is not None:
            if detected_frame and last_kp_raw is not None:
                draw_detected_skeleton(render, last_kp_raw,
                                       last_result["severities"], H, W)
            if (detected_frame
                    and last_kp_raw is not None
                    and last_result["pose_conf"] >= CONF_GATE):
                draw_ideal_skeleton(render, last_kp_raw,
                                    last_result["pose_id"],
                                    last_result["pose_conf"], H, W)
            draw_text_panel(render, last_result)

        writer.write(render)
        frame_idx += 1

    cap.release()
    writer.release()
    print(f"Done. Wrote {frame_idx} frames -> {video_out}")

print("run_yoga_inference ready")

# ========================= Batch runner =========================
INPUT_BASE  = "path/to/yognet7poses"
OUTPUT_BASE = "path/to/working"

videos = [
    ("Adhomukhasvanasana", "32.mp4"),
    ("Bhujangasana",       "56.mp4"),
    ("Shavasana",          "5.mp4"),
    ("Tadasana",           "6.mp4"),
    ("Trikonasana",        "48.mp4"),
    ("Virbhadrasana1",     "25.mp4"),
    ("Vrikshasana",        "10.mp4"),
]

for pose_folder, video_file in videos:
    video_in  = os.path.join(INPUT_BASE, pose_folder, video_file)
    video_out = os.path.join(OUTPUT_BASE,
                             f"{pose_folder}_{video_file.replace(' ', '_')}_synthpose_output.mp4")
    print(f"\nProcessing: {video_in} -> {video_out}")
    run_yoga_inference(video_in, video_out)
    print(f"Finished: {video_out}")


In [ ]:
# Stage 9 - wrap-up zip 1/2: results bundle (figures, checkpoint, metrics, CSVs, README).
import os, shutil, glob, zipfile, json, datetime
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, confusion_matrix, classification_report

BUNDLE_DIR = "path/to/working/_bundle_results"
ZIP_PATH   = "path/to/working/synthpose_ablation_results.zip"

shutil.rmtree(BUNDLE_DIR, ignore_errors=True)
os.makedirs(BUNDLE_DIR, exist_ok=True)

# ==================== Figures ====================
fig_src = "path/to/working/results"
fig_files = [
    "viz_ablation_quality_confusion.png",
    "viz_ablation_per_pose_f1.png",
    "viz_ablation_feedback_mae.png",
]
os.makedirs(f"{BUNDLE_DIR}/figures", exist_ok=True)
for f in fig_files:
    src = os.path.join(fig_src, f)
    if os.path.exists(src):
        shutil.copy(src, f"{BUNDLE_DIR}/figures/{f}")
        print(f"  figure: {f}")
    else:
        print(f"  [warn] missing figure: {src}")

# ==================== Model checkpoint ====================
os.makedirs(f"{BUNDLE_DIR}/model", exist_ok=True)
shutil.copy("path/to/working/yoga_tcn_synthpose.pt",
            f"{BUNDLE_DIR}/model/yoga_tcn_synthpose.pt")
print(f"  model: yoga_tcn_synthpose.pt")

# ==================== Metrics CSV (test set) ====================
os.makedirs(f"{BUNDLE_DIR}/metrics", exist_ok=True)

# Overall + per-class pose F1
pose_f1_arr = f1_score(all_p_true, all_p_preds, average=None,
                       labels=list(range(len(POSE_NAMES))))
per_pose_rows = []
for i, p in enumerate(POSE_NAMES):
    per_pose_rows.append({"pose": p, "f1": float(pose_f1_arr[i])})
pd.DataFrame(per_pose_rows).to_csv(f"{BUNDLE_DIR}/metrics/per_pose_f1.csv", index=False)

# Per-joint feedback MAE — corrupted only and all samples
corrupted_mask = (all_q_true == 1)
per_joint_rows = []
for j, name in enumerate(JOINT_NAMES):
    mae_corr = float(np.abs(all_fb_preds[corrupted_mask, j] -
                            all_fb_true[corrupted_mask, j]).mean())
    mae_all  = float(np.abs(all_fb_preds[:, j] - all_fb_true[:, j]).mean())
    per_joint_rows.append({"joint": name,
                           "mae_corrupted_deg": mae_corr,
                           "mae_all_deg":       mae_all})
pd.DataFrame(per_joint_rows).to_csv(f"{BUNDLE_DIR}/metrics/per_joint_feedback_mae.csv",
                                    index=False)

# Quality head confusion
q_cm = confusion_matrix(all_q_true, all_q_preds)
q_acc = q_cm.diagonal().sum() / q_cm.sum()
pd.DataFrame({
    "label":               ["correct", "incorrect"],
    "predicted_correct":   [int(q_cm[0, 0]), int(q_cm[1, 0])],
    "predicted_incorrect": [int(q_cm[0, 1]), int(q_cm[1, 1])],
    "recall":              [q_cm[0, 0] / q_cm[0].sum(),
                            q_cm[1, 1] / q_cm[1].sum()],
}).to_csv(f"{BUNDLE_DIR}/metrics/quality_confusion.csv", index=False)

# Summary JSON — one place the writeup can cite topline numbers
summary = {
    "topology":                 "synthpose-52",
    "test_quality_accuracy":    float(q_acc),
    "test_quality_recall_correct":   float(q_cm[0, 0] / q_cm[0].sum()),
    "test_quality_recall_incorrect": float(q_cm[1, 1] / q_cm[1].sum()),
    "test_pose_accuracy":       float((all_p_preds == all_p_true).mean()),
    "test_pose_f1_macro":       float(np.mean(pose_f1_arr)),
    "test_feedback_mae_corrupted_mean_deg":
        float(np.mean([r["mae_corrupted_deg"] for r in per_joint_rows])),
    "n_windows_test":           int(len(all_q_true)),
    "n_poses":                  int(N_POSES),
    "n_joints":                 int(N_JOINTS),
    "n_keypoints":              52,
    "feat_dim":                 int(FEAT_DIM),
    "window_size":              int(WINDOW_SIZE),
    "generated_utc":            datetime.datetime.utcnow().isoformat() + "Z",
}
with open(f"{BUNDLE_DIR}/metrics/summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("  metrics: per_pose_f1.csv, per_joint_feedback_mae.csv, "
      "quality_confusion.csv, summary.json")

# ==================== Metadata CSVs ====================
os.makedirs(f"{BUNDLE_DIR}/metadata", exist_ok=True)
# Stage 1′ extraction metadata
shutil.copy("path/to/working/extracted_yognet_synthpose/metadata.csv",
            f"{BUNDLE_DIR}/metadata/stage1_extraction.csv")
# Stage 2′ window split metadata
for split in ["train", "val", "test"]:
    shutil.copy(f"path/to/working/windows_synthpose/meta_{split}.csv",
                f"{BUNDLE_DIR}/metadata/stage2_meta_{split}.csv")
print("  metadata: stage1_extraction.csv, stage2_meta_{train,val,test}.csv")

# ==================== Stage 7′ inference videos ====================
os.makedirs(f"{BUNDLE_DIR}/videos", exist_ok=True)
video_files = glob.glob("path/to/working/*_synthpose_output.mp4")
for v in sorted(video_files):
    shutil.copy(v, f"{BUNDLE_DIR}/videos/{os.path.basename(v)}")
    print(f"  video: {os.path.basename(v)}")

# ==================== Training log ====================
# Stage 3′ epochs captured from stdout — write them to a .txt here.
# Replace the contents below with your actual stdout from Stage 3′ Cell 5 if needed.
STAGE3_PRIME_LOG = """Epoch 001 | loss=2.0533 | val_quality_acc=0.4944
Epoch 002 | loss=1.8643 | val_quality_acc=0.5149
Epoch 003 | loss=1.8414 | val_quality_acc=0.5028
Epoch 004 | loss=1.8171 | val_quality_acc=0.5345
Epoch 005 | loss=1.8216 | val_quality_acc=0.5251
Epoch 006 | loss=1.8072 | val_quality_acc=0.5372
Epoch 007 | loss=1.8049 | val_quality_acc=0.5233
Epoch 008 | loss=1.7781 | val_quality_acc=0.5149
Epoch 009 | loss=1.7621 | val_quality_acc=0.5633
Epoch 010 | loss=1.7433 | val_quality_acc=0.5624
Epoch 011 | loss=1.6919 | val_quality_acc=0.5335
Epoch 012 | loss=1.6609 | val_quality_acc=0.5410
Epoch 013 | loss=1.6528 | val_quality_acc=0.5242
Epoch 014 | loss=1.5929 | val_quality_acc=0.5782
Epoch 015 | loss=1.5381 | val_quality_acc=0.5410
Epoch 016 | loss=1.5091 | val_quality_acc=0.5410
Epoch 017 | loss=1.4708 | val_quality_acc=0.5158
Epoch 018 | loss=1.4373 | val_quality_acc=0.5531
Epoch 019 | loss=1.4036 | val_quality_acc=0.5456
Epoch 020 | loss=1.3795 | val_quality_acc=0.5298
Epoch 021 | loss=1.2964 | val_quality_acc=0.5456
Epoch 022 | loss=1.2554 | val_quality_acc=0.5577
Epoch 023 | loss=1.2361 | val_quality_acc=0.5456
Epoch 024 | loss=1.2338 | val_quality_acc=0.5363  (early stop)
Best val quality accuracy: 0.5782
"""
os.makedirs(f"{BUNDLE_DIR}/logs", exist_ok=True)
with open(f"{BUNDLE_DIR}/logs/stage3_prime_training.txt", "w") as f:
    f.write(STAGE3_PRIME_LOG)
print("  log: stage3_prime_training.txt")

# ==================== README ====================
readme = f"""# SynthPose YogNet Ablation — Results Bundle

Generated: {summary['generated_utc']}

## What this is
Proposal 2 of the Yoga TCN project: swap BlazePose's 33-keypoint topology for
SynthPose's 52 anatomical keypoints (17 COCO + 35 bony markers from the
OpenCapBench paper) and retrain end-to-end on YogNet. Compares directly against
the BlazePose results from notebook `cs7150-video2`.

## Topline results (SynthPose, YogNet test set)
- Pose classification accuracy:   {summary['test_pose_accuracy']:.1%}
- Pose macro-F1:                  {summary['test_pose_f1_macro']:.3f}
- Quality binary accuracy:        {summary['test_quality_accuracy']:.1%}
- Quality recall (correct):       {summary['test_quality_recall_correct']:.1%}
- Quality recall (incorrect):     {summary['test_quality_recall_incorrect']:.1%}
- Feedback MAE (mean over 6 joints, corrupted samples):
                                  {summary['test_feedback_mae_corrupted_mean_deg']:.2f}°

## Headline finding
Mixed ablation. Pose head is near-parity with BlazePose (93.7% vs 96%).
Feedback head improves on 3/6 joints — notably both knees drop MAE by
30-50% thanks to medial+lateral knee markers — but hips regress (ASIS
is anterior to the true hip rotation center). Quality head regresses
substantially (91.5% → 55.7%), attributable to loss of 3DYoga90
pretraining; SynthPose trained on YogNet-only (~280 clips) without
the 3DYoga90 scale advantage BlazePose had.

## Directory layout
- figures/   — the three ablation comparison PNGs
- model/     — Stage 3' TCN checkpoint bundle (PyTorch .pt)
- metrics/   — per-class/per-joint metric CSVs + summary.json
- metadata/  — extraction and train/val/test window split CSVs
- videos/    — Stage 7' inference output videos with skeleton overlay
- logs/      — Stage 3' training stdout

## Reproducing
- Raw features (.npy clips, windowed arrays) are in the companion zip
  `synthpose_data.zip` — too large to bundle here.
- Raw videos: YogNet 7poses Kaggle dataset (yognet7poses).
- Models: stanfordmimi/synthpose-vitpose-base-hf + PekingU/rtdetr_r50vd_coco_o365
  (both on HuggingFace).

## BlazePose comparison source
Numbers for the BlazePose arm of the ablation are hardcoded in the Stage 8
figures from `cs7150-video2` notebook cells 57/58/59 (Stage 6 YogNet test).
"""
with open(f"{BUNDLE_DIR}/README.md", "w") as f:
    f.write(readme)
print("  README.md")

# ==================== Zip it ====================
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(BUNDLE_DIR):
        for f in files:
            full = os.path.join(root, f)
            arc  = os.path.relpath(full, BUNDLE_DIR)
            z.write(full, arc)

zip_mb = os.path.getsize(ZIP_PATH) / 1024 / 1024
print(f"\n✅ Results bundle: {ZIP_PATH}  ({zip_mb:.1f} MB)")


In [ ]:
# Stage 9 - wrap-up zip 2/2: raw data bundle (all .npy clips + windowed arrays).
import os, zipfile, datetime

DATA_ZIP = "path/to/working/synthpose_data.zip"

# Everything we want in the data zip
DATA_SOURCES = [
    # Stage 1′ — (T, 52, 3) per-clip arrays + metadata
    ("path/to/working/extracted_yognet_synthpose", "extracted_yognet_synthpose"),
    # Stage 2′ — windowed feature tensors, labels, feature stats, split metadata
    ("path/to/working/windows_synthpose",          "windows_synthpose"),
]

# Informative README for the data zip
DATA_README = f"""# SynthPose YogNet — Raw Data Bundle

Generated: {datetime.datetime.utcnow().isoformat()}Z

## What this is
All intermediate arrays produced by Stage 1' (SynthPose keypoint extraction)
and Stage 2' (feature engineering + augmentation + windowing) of the
SynthPose ablation notebook. These files let anyone retrain Stage 3'
without re-running extraction.

## Contents
- extracted_yognet_synthpose/
    clips/*.npy         — per-clip (T, 52, 3) arrays: (x_px, y_px, score)
    metadata.csv        — one row per clip (pose, quality, frames, paths)

- windows_synthpose/
    X_{{train,val,test}}.npy      — (N, 32, 272) windowed features
    y_quality_*.npy              — (N,) binary labels
    y_pose_*.npy                 — (N,) integer pose labels 0..6
    y_feedback_*.npy             — (N, 6) degree-corruption targets
    feature_stats.npz            — train-set feature mean/std for standardization
    meta_*.csv                   — one row per window, full provenance

## Schema notes
- Clip arrays use raw pixel coordinates (not normalized).
- Windowed X tensors are NOT standardized — apply (X - feat_mean)/feat_std
  at training time. See Stage 3'-2 Dataset class for the canonical pattern.
- Feature layout (D=272):
    [0..103]    xy        (52 kp * 2, hip-centered and shoulder-scaled)
    [104..207]  vxy       (first-difference of xy across time)
    [208..259]  score     (per-keypoint detection confidence)
    [260..271]  angles    (6 joint angles + 3 L-R diffs + 3 L+R means)
"""

if not os.path.exists(DATA_SOURCES[0][0]):
    raise FileNotFoundError(
        f"Expected Stage 1' outputs at {DATA_SOURCES[0][0]}; run Stage 1' first")
if not os.path.exists(DATA_SOURCES[1][0]):
    raise FileNotFoundError(
        f"Expected Stage 2' outputs at {DATA_SOURCES[1][0]}; run Stage 2' first")

with zipfile.ZipFile(DATA_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    total_files = 0
    for src_dir, arc_base in DATA_SOURCES:
        for root, _, files in os.walk(src_dir):
            for f in files:
                full = os.path.join(root, f)
                rel  = os.path.relpath(full, src_dir)
                arc  = os.path.join(arc_base, rel)
                z.write(full, arc)
                total_files += 1
    # Embed the data README at archive root
    z.writestr("README.md", DATA_README)
    total_files += 1

zip_mb = os.path.getsize(DATA_ZIP) / 1024 / 1024
print(f"✅ Data bundle: {DATA_ZIP}  ({zip_mb:.1f} MB, {total_files} files)")
